In [1]:
### import libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from nltk.sentiment import SentimentIntensityAnalyzer

In [2]:
# read the dataset
df = pd.read_csv("data/test.csv")
df.shape

(2191, 4)

In [3]:
df['body']

0       EnronOptions Announcement\r\n\r\n\r\nWe have u...
1       Marc,\r\n\r\nUnfortunately, today is not going...
2       When: Wednesday, June 06, 2001 10:00 AM-11:00 ...
3       we were thinking papasitos (we can meet somewh...
4       Since you never gave me the $20 for the last t...
                              ...                        
2186    Thanks for the resume.  She has had some good ...
2187    Attached please find the following documents:\...
2188    Good to finally hear from.  Judging from your ...
2189    It looks like we have our 12 teams.  We will p...
2190    We will need this, so I am sending it to you a...
Name: body, Length: 2191, dtype: object

In [4]:
## delete '\n' string inside the message
df['body'] = df['body'].str.replace('\n', ' ')
df['body'] = df['body'].str.replace(r'\s+', ' ', regex=True).str.strip()
df['body'] = df['body'].str.replace('=20', ' ', regex=False)
df['body'] = df['body'].str.replace('=01', '', regex=False)

In [5]:
df['body']

0       EnronOptions Announcement We have updated the ...
1       Marc, Unfortunately, today is not going to wor...
2       When: Wednesday, June 06, 2001 10:00 AM-11:00 ...
3       we were thinking papasitos (we can meet somewh...
4       Since you never gave me the $20 for the last t...
                              ...                        
2186    Thanks for the resume. She has had some good e...
2187    Attached please find the following documents: ...
2188    Good to finally hear from. Judging from your e...
2189    It looks like we have our 12 teams. We will pr...
2190    We will need this, so I am sending it to you a...
Name: body, Length: 2191, dtype: object

### Task 1: Sentiment Labeling

In [6]:
analyzer = SentimentIntensityAnalyzer()

sentiment_label = []
for sentence in df['body']:
    vs = analyzer.polarity_scores(sentence)
    score = vs['compound']
    if score >= 0.05:
        sentiment_label.append('Positive')
    elif score <= -0.05:
        sentiment_label.append('Negative')
    else:
        sentiment_label.append('Neutral')

In [7]:
df['Sentiment'] = sentiment_label
df[['body','Sentiment']]

,body,Sentiment
0,EnronOptions Announcement We have updated the ...,Positive
1,"Marc, Unfortunately, today is not going to wor...",Positive
2,"When: Wednesday, June 06, 2001 10:00 AM-11:00 ...",Neutral
3,we were thinking papasitos (we can meet somewh...,Neutral
4,Since you never gave me the $20 for the last t...,Positive
...,...,...
2186,Thanks for the resume. She has had some good e...,Positive
2187,Attached please find the following documents: ...,Positive
2188,Good to finally hear from. Judging from your e...,Positive
2189,It looks like we have our 12 teams. We will pr...,Positive


In [8]:
print(f"The number of positive message(s): {(df['Sentiment'] == 'Positive').sum()}")
print(f"The number of neutral message(s): {(df['Sentiment'] == 'Neutral').sum()}")
print(f"The number of negative message(s): {(df['Sentiment'] == 'Negative').sum()}")

The number of positive message(s): 1528
The number of neutral message(s): 511
The number of negative message(s): 152


### Task 2 : EDA

In [9]:
print(f"There are {df.shape[0]} emails in this dataset.")

There are 2191 emails in this dataset.


In [10]:
# Create a variable to represent year only
df['Year'] = pd.to_datetime(df['date']).dt.year
df.head(5)

,Subject,body,date,from,Sentiment,Year
0,EnronOptions Update!,EnronOptions Announcement We have updated the ...,5/10/2010,sally.beck@enron.com,Positive,2010
1,(No Subject),"Marc, Unfortunately, today is not going to wor...",7/29/2010,eric.bass@enron.com,Positive,2010
2,Phone Screen Interview - Shannon L. Burnham,"When: Wednesday, June 06, 2001 10:00 AM-11:00 ...",7/25/2011,sally.beck@enron.com,Neutral,2011
3,RE: My new work email,we were thinking papasitos (we can meet somewh...,3/25/2010,johnny.palmer@enron.com,Neutral,2010
4,Bet,Since you never gave me the $20 for the last t...,5/21/2011,lydia.delgado@enron.com,Positive,2011


In [11]:
# Check the numbers of emails each year
df['Year'].value_counts()

Year
2011    1097
2010    1094
Name: count, dtype: int64

In [12]:
duplication = df.duplicated(subset=['body'])
print(f"There are {len(df[duplication])} duplicates email bodies.")

There are 682 duplicates email bodies.


In [13]:
df['from'].nunique()
## There are only 10 unique senders from 2010 to 2011

10

### Task 3: Employee Score Calculation

In [14]:
"""
24 months 
10 employees
monthly score of 10 employees based on the message (df['body])

Approach: 
Column : Monthly Sentiment Score
Calculate the sentiment score based on df['Sentiment']
        if df['Sentiment'] == 'positive' -> df['Monthly Sentiment Score'] = 1
        else if df['Sentiment'] == 'negative --> df['Monthly Sentiment Score'] = -1
        else: df['Monthly Sentiment Score'] = 0 
Make a "MM-YYYY" column -> there would be 24 unique months
Calculate score of each employee each month = group by month -> then group by employee -> add df["Monthly Sentiment Score']
--> Ensure that the score resets at the beginning of each new month
"""

'\n24 months \n10 employees\nmonthly score of 10 employees based on the message (df[\'body])\n\nApproach: \nColumn : Monthly Sentiment Score\nCalculate the sentiment score based on df[\'Sentiment\']\n        if df[\'Sentiment\'] == \'positive\' -> df[\'Monthly Sentiment Score\'] = 1\n        else if df[\'Sentiment\'] == \'negative --> df[\'Monthly Sentiment Score\'] = -1\n        else: df[\'Monthly Sentiment Score\'] = 0 \nMake a "MM-YYYY" column -> there would be 24 unique months\nCalculate score of each employee each month = group by month -> then group by employee -> add df["Monthly Sentiment Score\']\n--> Ensure that the score resets at the beginning of each new month\n'

In [15]:
# Set score for each sentiment
score = {'Positive': 1, 'Neutral': 0, 'Negative': -1}

# Assign a score to each message
df['Score'] = df['Sentiment'].map(score)
df.head(10)

,Subject,body,date,from,Sentiment,Year,Score
0,EnronOptions Update!,EnronOptions Announcement We have updated the ...,5/10/2010,sally.beck@enron.com,Positive,2010,1
1,(No Subject),"Marc, Unfortunately, today is not going to wor...",7/29/2010,eric.bass@enron.com,Positive,2010,1
2,Phone Screen Interview - Shannon L. Burnham,"When: Wednesday, June 06, 2001 10:00 AM-11:00 ...",7/25/2011,sally.beck@enron.com,Neutral,2011,0
3,RE: My new work email,we were thinking papasitos (we can meet somewh...,3/25/2010,johnny.palmer@enron.com,Neutral,2010,0
4,Bet,Since you never gave me the $20 for the last t...,5/21/2011,lydia.delgado@enron.com,Positive,2011,1
5,RE: Favor,"sure, just call me the bank that delivers. we ...",10/23/2011,eric.bass@enron.com,Positive,2011,1
6,MG Inventory Summaries,Inventory summaries for both MGL and MGMCC as ...,4/5/2010,kayne.coulter@enron.com,Neutral,2010,0
7,Forgot the Attachment,Please print attachment and make sure that e:m...,4/21/2010,patti.thompson@enron.com,Positive,2010,1
8,Garvin Brown - AXIA Sr. Power Scheduler,Please advise me of your interest in Garvin's ...,2/7/2010,sally.beck@enron.com,Positive,2010,1
9,More Dallas ASE Information,The start time for Tuesday morning has been ch...,2/6/2010,kayne.coulter@enron.com,Negative,2010,-1


In [16]:
df['date'] = pd.to_datetime(df['date'])
df['Month'] = df['date'].dt.to_period('M')
df['Month']

0       2010-05
1       2010-07
2       2011-07
3       2010-03
4       2011-05
         ...   
2186    2011-06
2187    2011-01
2188    2011-01
2189    2011-03
2190    2010-10
Name: Month, Length: 2191, dtype: period[M]

In [17]:
monthly_scores = df.groupby(['Month', 'from'])['Score'].sum()
monthly_scores = monthly_scores.reset_index()
monthly_scores = monthly_scores.rename(columns={'from': 'Employee', 'Score': 'Monthly Sentiment Score'})
monthly_scores.head(20)

,Month,Employee,Monthly Sentiment Score
0,2010-01,bobette.riner@ipgdirect.com,1
1,2010-01,don.baughman@enron.com,5
2,2010-01,eric.bass@enron.com,9
3,2010-01,john.arnold@enron.com,5
4,2010-01,johnny.palmer@enron.com,1
5,2010-01,kayne.coulter@enron.com,13
6,2010-01,lydia.delgado@enron.com,9
7,2010-01,patti.thompson@enron.com,6
8,2010-01,rhonda.denton@enron.com,1
9,2010-01,sally.beck@enron.com,1


In [18]:
monthly_scores['Month'].unique()
# 2 years = 24 months (2010-2011)

<PeriodArray>
['2010-01', '2010-02', '2010-03', '2010-04', '2010-05', '2010-06', '2010-07',
 '2010-08', '2010-09', '2010-10', '2010-11', '2010-12', '2011-01', '2011-02',
 '2011-03', '2011-04', '2011-05', '2011-06', '2011-07', '2011-08', '2011-09',
 '2011-10', '2011-11', '2011-12']
Length: 24, dtype: period[M]

In [19]:
for employee in monthly_scores['Employee'].unique():
    print(f"{employee}")
    employee_name = monthly_scores[monthly_scores['Employee'] == employee]
    print(employee_name[['Month', 'Monthly Sentiment Score']].head(5))

bobette.riner@ipgdirect.com
      Month  Monthly Sentiment Score
0   2010-01                        1
10  2010-02                        7
20  2010-03                        6
30  2010-04                        3
40  2010-05                        2
don.baughman@enron.com
      Month  Monthly Sentiment Score
1   2010-01                        5
11  2010-02                        6
21  2010-03                        2
31  2010-04                        9
41  2010-05                       16
eric.bass@enron.com
      Month  Monthly Sentiment Score
2   2010-01                        9
12  2010-02                        2
22  2010-03                        4
32  2010-04                        2
42  2010-05                        6
john.arnold@enron.com
      Month  Monthly Sentiment Score
3   2010-01                        5
13  2010-02                       11
23  2010-03                        7
33  2010-04                        8
43  2010-05                        3
johnny.palmer@enron

In [20]:
monthly_scores[monthly_scores['Monthly Sentiment Score'] < 0]

,Month,Employee,Monthly Sentiment Score
132,2011-02,eric.bass@enron.com,-1
175,2011-06,kayne.coulter@enron.com,-1
186,2011-07,lydia.delgado@enron.com,-1


### Task 4: Employee Ranking

In [45]:
positive_scores = monthly_scores.sort_values(by=['Month', 'Monthly Sentiment Score', 'Employee'], ascending=[True, False, True])
top_three_positive = positive_scores.groupby('Month').head(3).reset_index(drop=True)
top_three_positive

,Month,Employee,Monthly Sentiment Score
0,2010-01,kayne.coulter@enron.com,13
1,2010-01,eric.bass@enron.com,9
2,2010-01,lydia.delgado@enron.com,9
3,2010-02,john.arnold@enron.com,11
4,2010-02,johnny.palmer@enron.com,10
...,...,...,...
67,2011-11,john.arnold@enron.com,10
68,2011-11,bobette.riner@ipgdirect.com,9
69,2011-12,eric.bass@enron.com,12
70,2011-12,patti.thompson@enron.com,12


In [59]:
for month in top_three_positive['Month'].unique() :
    print(f'Month: {month}')
    top_positive = top_three_positive[top_three_positive['Month'] == month]
    print(top_positive[['Employee', 'Monthly Sentiment Score']].reset_index(drop=True))

Month: 2010-01
                  Employee  Monthly Sentiment Score
0  kayne.coulter@enron.com                       13
1      eric.bass@enron.com                        9
2  lydia.delgado@enron.com                        9
Month: 2010-02
                      Employee  Monthly Sentiment Score
0        john.arnold@enron.com                       11
1      johnny.palmer@enron.com                       10
2  bobette.riner@ipgdirect.com                        7
Month: 2010-03
                      Employee  Monthly Sentiment Score
0         sally.beck@enron.com                       11
1        john.arnold@enron.com                        7
2  bobette.riner@ipgdirect.com                        6
Month: 2010-04
                  Employee  Monthly Sentiment Score
0   don.baughman@enron.com                        9
1  kayne.coulter@enron.com                        9
2    john.arnold@enron.com                        8
Month: 2010-05
                   Employee  Monthly Sentiment Score
0    don

In [54]:
lowest_scores = monthly_scores.sort_values(by=['Month', 'Monthly Sentiment Score', 'Employee'], ascending=[True, True, True])
top_three_lowest = lowest_scores.groupby('Month').head(3).reset_index(drop=True)
top_three_lowest

,Month,Employee,Monthly Sentiment Score
0,2010-01,bobette.riner@ipgdirect.com,1
1,2010-01,johnny.palmer@enron.com,1
2,2010-01,rhonda.denton@enron.com,1
3,2010-02,kayne.coulter@enron.com,1
4,2010-02,lydia.delgado@enron.com,1
...,...,...,...
67,2011-11,rhonda.denton@enron.com,2
68,2011-11,don.baughman@enron.com,5
69,2011-12,johnny.palmer@enron.com,2
70,2011-12,bobette.riner@ipgdirect.com,3


In [60]:
for month in top_three_lowest['Month'].unique() :
    print(f'Month: {month}')
    top_lowest = top_three_lowest[top_three_lowest['Month'] == month]
    print(top_lowest[['Employee', 'Monthly Sentiment Score']].reset_index(drop=True))

Month: 2010-01
                      Employee  Monthly Sentiment Score
0  bobette.riner@ipgdirect.com                        1
1      johnny.palmer@enron.com                        1
2      rhonda.denton@enron.com                        1
Month: 2010-02
                   Employee  Monthly Sentiment Score
0   kayne.coulter@enron.com                        1
1   lydia.delgado@enron.com                        1
2  patti.thompson@enron.com                        1
Month: 2010-03
                  Employee  Monthly Sentiment Score
0  rhonda.denton@enron.com                        1
1   don.baughman@enron.com                        2
2  kayne.coulter@enron.com                        3
Month: 2010-04
                      Employee  Monthly Sentiment Score
0          eric.bass@enron.com                        2
1  bobette.riner@ipgdirect.com                        3
2         sally.beck@enron.com                        3
Month: 2010-05
                      Employee  Monthly Sentiment Score
0